# MarkItDown vs. Docling for PDF

`l22_document_loaders.ipynb` covers plain-text PDF loaders (`PyPDFLoader`, `PyMuPDFLoader`), and
`l22_chunking_strategies.ipynb` already uses Docling — but only for its `HybridChunker`, never for
Docling's own core job: converting a PDF straight to structured **Markdown**. This notebook
compares two Markdown-first PDF converters head-to-head:

1. **MarkItDown** — Microsoft's general file-to-Markdown utility (pptx, docx, xlsx, pdf, images,
   audio, and more, all through one API).
2. **Docling** — already a course dependency, used here for `DocumentConverter → export_to_markdown()`
   instead of chunking.

Neither tool calls an LLM for plain PDF conversion in this notebook. MarkItDown does have an
LLM-enhanced mode, but per its own documentation that currently applies only to **pptx and image
files, not PDF** — so it's not used here. The only network call anywhere below, if any, would be
the same OpenAI embeddings call already established elsewhere in this course, used only in the
brief pipeline tie-in at the end.

Two source PDFs, chosen for a reason: `data/sampled_schemes.pdf` (prose and headings, **no
tables** — used throughout the `l22_*` notebooks) for a general comparison, and
`data/ai_summit.pdf` (confirmed via Docling's own `doc.tables` to contain **exactly one real
table**) for the table-fidelity comparison.


In [ ]:
SCHEMES_PDF = "data/sampled_schemes.pdf"   # prose + headings, no tables
SUMMIT_PDF = "data/ai_summit.pdf"          # has one real table (Date | Event | Venue)


In [ ]:
uv add "markitdown[pdf]"

## 1. MarkItDown

```
pip install 'markitdown[pdf]'
```

The API is deliberately minimal: one class, one method, a `.text_content` string back.


In [ ]:
from markitdown import MarkItDown

markitdown_client = MarkItDown(enable_plugins=False)

md_schemes = markitdown_client.convert(SCHEMES_PDF).text_content
print(f"{len(md_schemes):,} chars")
print(md_schemes)


In [ ]:
md_summit = markitdown_client.convert(SUMMIT_PDF).text_content
print(f"{len(md_summit):,} chars")
print(md_summit)

Notice what's *missing* above: no `#`/`##` heading markers anywhere in `md_schemes`, even though
"CENTRALLY SPONSORED SCHEMES" and "NATIONAL FOOD SECURITY MISSION" are clearly section headings in
the source PDF (larger/bold text). This isn't a setting we forgot to pass — it's built into how the
installed converter works. Reading `markitdown/converters/_pdf_converter.py` directly, its `PdfConverter`
has exactly two output paths for a page's content, and neither one detects headings:

1. A form/table heuristic (`_extract_form_content_from_words`) that looks at word x/y positions to
   guess at table columns — covered in the table-fidelity section below.
2. A plain-text fallback via `pdfminer.high_level.extract_text()`, used for every page that doesn't
   trigger path 1.

There is no font-size check, no bold-weight check, no heading-role classification anywhere in that
file. For a prose document like `sampled_schemes.pdf`, no page trips the table heuristic, so the
*entire* document falls through to path 2 — plain `pdfminer` text with line breaks preserved, full
stop. No parameter turns headings on, because heading detection for PDFs simply isn't implemented in
this converter. Docling's layout-role model (used below) is a genuinely different approach — it
classifies each text block's role (title, section-header, body text, ...) before export — which is
why it's the tool to reach for when heading/structure preservation from a PDF actually matters.

On the LLM-enhanced mode: MarkItDown supports `MarkItDown(llm_client=OpenAI(), llm_model="gpt-4o")`
for richer descriptions of embedded images — but its own docs scope that to **pptx and image
files**, not PDF text extraction, so it wouldn't add headings to the conversions above even if
enabled.


## 2. Docling

Same environment note as `l22_chunking_strategies.ipynb`'s Docling section: this machine has no
C++ compiler, which breaks `torch.compile` inside Docling's layout model, so `TORCHDYNAMO_DISABLE=1`
is needed before importing it.


In [11]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import logging
logging.getLogger("docling").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions 
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions(do_ocr=False)
docling_converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

docling_schemes_doc = docling_converter.convert(SCHEMES_PDF).document
docling_md_schemes = docling_schemes_doc.export_to_markdown()

print(f"{len(docling_md_schemes):,} chars")
print(docling_md_schemes)


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 5504.23it/s]


16,492 chars
## CENTRALLY SPONSORED SCHEMES

## NATIONAL FOOD SECURITY MISSION

The scheme is being implemented from 2007 ­ 08 as a Central Sector Scheme with an objective to enhance production &amp; productivity of Rice &amp; Pulse crops in selected districts of the state in Mission Mode. Besides, this embodies Sub ­ Schemes for enhancing production &amp; productivity of Coarse Cereals under NFSM ­ Coarse Cereals, and Cotton, Jute and Sugarcane under NFSM ­ Commercial crops, NFSM ­ Nutricereals and NFSM ­ Oilseeds. Besides, Targeting Rice Fallow Areas is being implemented under NFSM for increasing coverage under pulses and oilseeds crops.

Objectives: Increase food security by stepping up the overall food production and food stocks held by the government; ensure the nation remains self ­ sufficient and prices remain under check.

Interventions: Under NFSM, the interventions are focused on increasing food production by organising:

-  Cluster Demonstrations -test beds for demonstratio

terventions: Under NFSM, the interventions are focused on increasing food production by organising:

-  Cluster Demonstrations -test beds for demonstrations are being set up on improved package and cropping systems.
-  Seed Distribution -Distribution of high ­ yielding varieties and hybrid seeds at subsidised costs.
-  Farm Machineries -Distribution of farm equipment and tools like conoweeder, manual sprayer, drum seeder, seed drill, zero till multi ­ crop planter, power weeder, paddy thresher, laser land leveller; all at subsidized costs.
-  Plant Protection -Subsidised distribution of plant protection chemicals and bio ­ pesticides; weedicides.
-  Micro ­ Nutrients and Soil Ameliorants -Distribution of subsidised micro ­ nutrients, lime/ liming materials, etc.
-  Resource conservation Tools and Techniques.
-  Local Initiatives -Five percent of total state allocation spent for funding on project basis, and
-  Setting up of project management teams at State and District levels.

Date Event Venue
AI Impact Expo Bharat Mandapam, New Delhi
16–20 February 2026
16 February 2026 Keynotes, Panel Discussions, Roundtables Bharat Mandapam/ Sushma Swaraj Bhawan/
Ambedkar Bhawan, New Delhi
Release of Knowledge Compendiums on AI in Bharat Mandapam, New Delhi
17 February 2026 Health, Energy, Education, Agriculture, Gender
Empowerment, Accessibility
Seminar on Applied AI
AI by HER: Global Impact Challenge Sushma Swaraj Bhawan, New Delhi
Keynotes, Panel Discussions, Roundtables Bharat Mandapam/ Sushma Swaraj Bhawan/
Ambedkar Bhawan, New Delhi
Research Symposium Bharat Mandapam, New Delhi
18 February 2026
Industry Session
AI by HER: Global Impact Challenge Sushma Swaraj Bhawan, New Delhi
Keynotes, Panel Discussions, Roundtables Bharat Mandapam/ Sushma Swaraj Bhawan/
Ambedkar Bhawan, New Delhi
Summit Dinner Convention Centre, New Delhi
Opening Ceremony
19 February 2026 Leaders’ Plenary Bharat Mandapam, New Delhi
CEO Roundtable
Keynotes / Panel Discussion/ Roundtables Bharat Mandapam / Sushma Swaraj Bhawan /
Ambedkar Bhawan, New Delhi
20 February 2026 GPAI Council Meeting Bharat Mandapam, New Delhi
Keynotes/ Panel Discussion/ Roundtables Bharat Mandapam / Sushma Swaraj Bhawan /
Ambedkar Bhawan, New Delhi

# Core Components of RAG

Retrieval-Augmented Generation (RAG) is a technique that combines **information retrieval with Large Language Models (LLMs)** to generate more accurate and context-aware answers. A typical RAG system has four core components: **document ingestion, retrieval, prompt augmentation, and generation**.

First, **document ingestion** collects data from sources such as PDFs, websites, databases, or documents. The data is cleaned and divided into smaller pieces called **chunks**. These chunks are converted into numerical representations called **embeddings** and stored in a vector database such as FAISS, Chroma, or Qdrant.

Second, the **retrieval component** searches the vector database when a user asks a question. The question is converted into an embedding, and the system finds the most relevant document chunks using similarity search. Hybrid search and reranking can also be used to improve retrieval accuracy.

Third, in **prompt augmentation**, the retrieved information is added to the user's question as context. This gives the LLM relevant external knowledge instead of relying only on its pretrained knowledge.

Finally, the **generation component** uses an LLM such as GPT, Claude, or Llama to generate the final answer based on the user's question and the retrieved context. This helps reduce hallucinations and allows the system to answer questions using private or domain-specific information.

In short, the RAG pipeline can be represented as:

**Documents → Chunking → Embeddings → Vector Database → Retrieval → Context + Query → LLM → Answer**

Thus, RAG provides a practical way to connect LLMs with external and frequently updated knowledge sources while improving the relevance and reliability of generated responses.



## CENTRALLY SPONSORED SCHEMES

## NATIONAL FOOD SECURITY MISSION

The scheme is being implemented from 2007 ­ 08 as a Central Sector Scheme with an objective to enhance production &amp; productivity of Rice &amp; Pulse crops in selected districts of the state in Mission Mode. Besides, this embodies Sub ­ Schemes for enhancing production &amp; productivity of Coarse Cereals under NFSM ­ Coarse Cereals, and Cotton, Jute and Sugarcane under NFSM ­ Commercial crops, NFSM ­ Nutricereals and NFSM ­ Oilseeds. Besides, Targeting Rice Fallow Areas is being implemented under NFSM for increasing coverage under pulses and oilseeds crops.

Objectives: Increase food security by stepping up the overall food production and food stocks held by the government; ensure the nation remains self ­ sufficient and prices remain under check.

Interventions: Under NFSM, the interventions are focused on increasing food production by organising:

-  Cluster Demonstrations -test beds for demonstrations are being set up on improved package and cropping systems.
-  Seed Distribution -Distribution of high ­ yielding varieties and hybrid seeds at subsidised costs.
-  Farm Machineries -Distribution of farm equipment and tools like conoweeder, manual sprayer, drum seeder, seed drill, zero till multi ­ crop planter, power weeder, paddy thresher, laser land leveller; all at subsidized costs.
-  Plant Protection -Subsidised distribution of plant protection chemicals and bio ­ pesticides; weedicides.
-  Micro ­ Nutrients and Soil Ameliorants -Distribution of subsidised micro ­ nutrients, lime/ liming materials, etc.
-  Resource conservation Tools and Techniques.
-  Local Initiatives -Five percent of total state allocation spent for funding on project basis, and
-  Setting up of project management teams at State and District levels.

## NATIONAL MISSION FOR SUSTAINABLE AGRICULTURE (NMSA)

The scheme is being implemented with different components viz. Soil Health Management (SHM), Soil Health Card, Paramparagat Krishi Vikas Yojana (PKVY), Bharatiya Prakrutik Krishi Padhatti (BPKP) etc through Directorate of agriculture.

## 1. NATIONAL PROJECT ON MANAGEMENT OF SOIL HEALTH AND FERTILITY ­ SOIL HEALTH CARD &amp; SOIL HEALTH MANAGEMENT (UNDER NMSA)

Objective: Provide farmers with soil health cards with recommendations based on soil test results for judicious and balanced fertiliser use and up ­ keeping soil health.

## Interventions:

-  Soil Health Card
-  Demon./ Fin. Assistance to Farmers
-  Farmers'Training
-  Campaign Mode One week Soil Health mela involving Farmers)
-  Workshop
-  ICT

## 2. PARAMPARAGAT KRISHI VIKAS YOJANA (PKVY)

Objective: Organic agriculture is a production of agricultural products free from chemicals and pesticides residues by adopting eco ­ friendly low cost technologies 'Param pragat Krishi Vikas Yojana' is an elaborated component of Soil Health Management (SHM) of National Mission of Sustainable Agriculture (NMSA). Under PKVY, Organic farming is promoted through adoption of organic village by cluster approach and PGS certification.

## Interventions:

- o Adoption &amp; PGS Certification through cluster approach
- o Mobilization of farmers/ local people to form cluster in 50 acre for PGS Certification

- o PGS Certification and Quality Control
- o Adoption of organic village for manure Management and biological nitrogen harvesting through cluster approach.
- o Action Plan for organic farming for one Cluster.
- o Integrated Manure Management
- o Custom Hiring centre (C.H.C) Charges
- o Packing, Labelling &amp; Branding of Organic products of cluster.

## NATIONAL MISSION ON AGRICULTURAL EXTENSION AND TECHNOLOGY (NMAET) :

The scheme is being implemented with four Sub ­ Missions viz. Sub Mission on Farm Mechanisation, Sub Mission on Agriculture Extension &amp; National e ­ Governance Plan, Sub Mission on Seed and Planting Materials and Sub Mission on Plant Protection and Quarantine.

Objective: The aim of the Mission is to restructure and strengthen agricultural extension to enable delivery of appropriate technology and improved agronomic practices to farmers.

## 1. SUB ­ MISSION ON AGRICULTURE MECHANISATION (SMAM) :

Objectives: The prime objective would be to promote agriculture mechanization through extending procurement subsidy &amp; establishment of Farm Machinery banks to facilitate custom hiring.

Interventions: Provision of subsidy to farmers on the selected improved farm machineries &amp; equipment's used for various agricultural operations (both during crop stand &amp; post ­ harvest), conducting training &amp; demonstrations on their use and maintenance. Besides, assistance would be provided for mechanized land preparation, establishment of farm machinery banks for custom hiring, High ­ Tech productive equipment Centres etc. Selected village(s) would be adopted for complete mechanization package.

## 2. SUB ­ MISSION ON SEED AND PLANTING MATERIAL (SMSP)

Objective: The scheme is being implemented with the prime objective of making available quality seeds and planting material to farmers through quality seed production, distribution and quality control.

Interventions : Promotion of seed production, quality seed distribution, capacity building on seed production, Infrastructure strengthening for quality seed production, processing and distribution etc.

## 3. SUB ­ MISSION ON AGRICULTURE EXTENSION (SMAE)

Objective: Sub ­ Mission on Agricultural Extension will focus on awareness creation and enhanced use of appropriate technologies in agriculture &amp; allied sectors through PRA studies besides promoting demand driven extension and promotion of market ­ led technologies.

Intervention: Gains made in the past will be consolidated and strengthened through increased penetration of extension functionaries. Personnel trained under Agri ­ Clinics and Agri ­ Business Centres Scheme (ACABC) and Diploma in Agriculture Extension Services for Input Dealers (DAESI) will also provide extension services to the farmers. Use of interactive and innovative methods of information dissemination like pico projectors, low cost films, handheld devices, mobile based services, Kisan Call Centres (KCCs) etc. will be used and convergence brought among extension efforts under different programmes and schemes at village level through the institution of ATMA (Agriculture Technology Management Agency) and Block Technology Teams (BTTs).

## 4. NATIONAL e ­ GOVERNANCE PLAN (NeGP)

NeGP takes a holistic view of e ­ governance initiatives across the country, integrating them into a collective vision, a shared cause. Around this idea, a massive countrywide infrastructure reaching down to the remotest of villages is evolving, and large ­ scale digitization of records is taking place to enable easy, reliable access over the internet.

Objective: The ultimate objective is to bring public services closer home to citizens. "Make all government services accessible to the common man in his locality, through common service delivery outlets, and ensure efficiency, transparency, and reliability of such services at affordable costs to realise the basic needs of the common man"

Intervention: Increased use of ICT in agriculture through networking every nook and corner up to grass root level.

## 5. SUB ­ MISSION ON PLANT PROTECTION AND QUARANTINE (SMPP)

Objective: Sub ­ Mission on Plant Protection included in NMAET envisages increase in agricultural production by keeping the crop disease free using scientific and environment friendly techniques through promotion of Integrated Pest Management. Strengthening and Modernization of Pest Management Approach aims at this vital aspect of Plant Protection and also covers regulatory requirements of pesticides. The component on Strengthening &amp; Modernization of Plant Quarantine facilities in India is regulatory in nature with the aim of preventing introduction and spread of exotic pests that are harmful to crops by regulating/restricting import of plant/plant products. Monitoring pesticide residues in food commodities and environmental samples is also included in this Sub ­ Mission.

Intervention: The scheme is implemented with the prime objective of modernising pest management by assisting activities on Plant protection and taking up quarantine measures at entry points like Airport/ Sea Port etc. to protect the state from invasion by unwanted organisms.

## Pradhan Mantri Fasal Bima Yojana (PMFBY)

## 1. Objective of the Scheme

Pradhan Mantri Fasal Bima Yojana (PMFBY) aims at supporting sustainable production in agriculture sector by way of
# Core Components of RAG

Retrieval-Augmented Generation (RAG) is a technique that combines **information retrieval with Large Language Models (LLMs)** to generate more accurate and context-aware answers. A typical RAG system has four core components: **document ingestion, retrieval, prompt augmentation, and generation**.

First, **document ingestion** collects data from sources such as PDFs, websites, databases, or documents. The data is cleaned and divided into smaller pieces called **chunks**. These chunks are converted into numerical representations called **embeddings** and stored in a vector database such as FAISS, Chroma, or Qdrant.

Second, the **retrieval component** searches the vector database when a user asks a question. The question is converted into an embedding, and the system finds the most relevant document chunks using similarity search. Hybrid search and reranking can also be used to improve retrieval accuracy.

Third, in **prompt augmentation**, the retrieved information is added to the user's question as context. This gives the LLM relevant external knowledge instead of relying only on its pretrained knowledge.

Finally, the **generation component** uses an LLM such as GPT, Claude, or Llama to generate the final answer based on the user's question and the retrieved context. This helps reduce hallucinations and allows the system to answer questions using private or domain-specific information.

In short, the RAG pipeline can be represented as:

**Documents → Chunking → Embeddings → Vector Database → Retrieval → Context + Query → LLM → Answer**

Thus, RAG provides a practical way to connect LLMs with external and frequently updated knowledge sources while improving the relevance and reliability of generated responses.

-  Providingfinancialsupporttofarmerssufferingcroploss/damagearisingouto funforeseenevents
-  Stabilizing the income of farmers to ensure their continuance in farming

-  Encouragingfarmerstoadoptinnovativeandmodernagriculturalpractices
-  Ensuringflowofcredittotheagriculturesectorwhichwillcontributetofoodse curity, crop diversification and enhancing growth and competitiveness of agriculture sector besides protecting farmers from production risks.

## 2. Adoption of Technology for Scheme Administration :

2.1

In an endeavour to integrate Technology in implementation and execution of the Scheme, the Govt. of India has designed and developed a National Crop Insurance Portal (NCIP) (www. pmfby.gov.in). This will bring in better administration and coordination amongst stakeholders viz. Farmers, States, Insurers and Banks as well as real ­ time dissemination of information and transparency.

2.2

2.3

2.4

2.5

The successful running of the Portal calls for responsible participation by different stakeholders who will have the responsibility for census coding and updating revenue/administrative units, AWS code mapping and updating requisite information/details as per login credential module.

Implementing States and Insurance Companies during each crop season are required to digitize and upload on the web Portal in the relevant module, basic information like notified areas, crops, sum insured, Govt. subsidy, and premium to be paid by farmers and name of the implementing Insurance Companies in the particular insurance unit etc., well within the prescribed time. This will facilitate farmers and other stakeholders to get the relevant information on Internet and through SMS. State Govt. and concerned Insurance Company will be responsible for any incorrect entry/ errors/ omissions etc.

Digitizationofbasicinformation/notificationshouldcompulsorilybedonebe forefloatingtenderdocuments which will be followed by entry of bided Premium rates and name of selected Insurance Company immediately after finalization of bids and issue of work order.

SincetheNationalCropInsurancePortalhasbeenconceptualisedforautoad

2.6

2.7

2.8

2.9

2.10

2.11

- 2.12

ministrationandseamless flow of data/information/reports on real time basis, State Govt. would not be allowed to create/use separate Portal/website for Crop Insurance purposes.

All Stakeholders have defined roles and responsibilities and accessibility to related modules on the Portal for administration of the Scheme. Details of operationalization of modules for each stakeholder are available on the Portal for ready reference.

Secured credential/ login preferably linked with Aadhar Number and mobile OTP based, for all Stakeholders viz, Central Government, State Governments, Banks, empanelled Insurance Companies and their designated field functionaries will be provided on the Portal to enable them to enter/upload/download the requisite information.

Insurance Companies shall not distribute/collect/allow any other proforma /utility/web Portal etc. for collecting details of insured farmers separately. However they may provide all requisite support to facilitate Bank Branches/ PACS for uploading the farmer's details on the Portal well within the prescribed cut ­ off dates.

Only farmers whose data is uploaded on the National Crop Insurance Portal shall be eligible for Insurance coverage and the premium subsidy from State and Central Govt. will be released accordingly.

All data pertaining to crop ­ wise, area ­ wise historical yield data, weather data, sown area, coverage and lames data, calamity years and actual yield shall be made available on the National Crop Insurance Portal for the purpose of premium rating, claim calculation etc.

Banks/Financial Institutions/other intermediaries need to compulsorily transfer the individual farmer's data electronically to the National Crop Insurance Portal. Accordingly Banks/FIs may endeavour to undertake CBS integration in a time bound manner for real time transfer of information/data.

It is also proposed to develop an integrated platform/portal for both PMFBY and Interest Subvention Scheme. The data/information of both the Schemes shall be auto synchronized to enable real time sharing of information and better program monitoring.

- 2.13 Insurance Companies shall compulsorily use technology/mobile applications for monitoring of crop health/Crop Cutting Experiments (CCEs) in coordination with concerned States. States shall also facilitate Insurance Companies with Satellite Imagery/Usage of Drones by way of prior approval of agency from which such data can be sourced. This is required for better monitoring and ground ­ truthing. ICs can be active partners in facilitating use of technology.
- 2.14 States shall adopt technology, such as satellite and UAV remote sensing, for various applications such as crop area estimation and yield disputes and also promote the use of remote sensing and other related technology for CCE planning, yield estimation, loss assessment, and assessment of prevented sowing and clustering of districts.

## 3. Coverage of Farmers

- 3.1 All farmers including sharecroppers and tenant farmers growing the notified crops in the notified areas are eligible for coverage. However, farmers should have insurable interest for the notified/insured crops. Then on ­ loanee farmers are required to submit necessary documentary evidence of land records prevailing in the State (Records of Right (RoR), Land possession Certificate (LPC)etc.)and/or applicable contract/ agreement details/ other documents notified/ permitted by concerned State Govt. in case of sharecroppers/tenant farmers and the same should be defined by the respective States in the notification itself.

## 3.1.1 Compulsory Component
| Date                  | Event                                                                                                               | Venue                                                             |
|-----------------------|---------------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------|
| 16 - 20 February 2026 | AI Impact Expo                                                                                                      | Bharat Mandapam, New Delhi                                        |
| 16 February 2026      | Keynotes, Panel Discussions, Roundtables                                                                            | Bharat Mandapam/ Sushma Swaraj Bhawan/ Ambedkar Bhawan, New Delhi |
| 17 February 2026      | Release of Knowledge Compendiums on AI in Health, Energy, Education, Agriculture, Gender Empowerment, Accessibility | Bharat Mandapam, New Delhi                                        |
| 17 February 2026      | Seminar on Applied AI                                                                                               | Bharat Mandapam, New Delhi                                        |
| 17 February 2026      | AI by HER: Global Impact Challenge                                                                                  | Sushma Swaraj Bhawan, New Delhi                                   |
| 17 February 2026      | Keynotes, Panel Discussions, Roundtables                                                                            | Bharat Mandapam/ Sushma Swaraj Bhawan/ Ambedkar Bhawan, New Delhi |
| 18 February 2026      | Research Symposium                                                                                                  | Bharat Mandapam, New Delhi                                        |
| 18 February 2026      | Industry Session                                                                                                    | Bharat Mandapam, New Delhi                                        |
| 18 February 2026      | AI by HER: Global Impact Challenge                                                                                  | Sushma Swaraj Bhawan, New Delhi                                   |
| 18 February 2026      | Keynotes, Panel Discussions, Roundtables                                                                            | Bharat Mandapam/ Sushma Swaraj Bhawan/ Ambedkar Bhawan, New Delhi |
| 18 February 2026      | Summit Dinner                                                                                                       | Convention Centre, New Delhi                                      |
| 19 February 2026      | Opening Ceremony Leaders' Plenary CEO Roundtable                                                                    | Bharat Mandapam, New Delhi                                        |
| 19 February 2026      | Keynotes / Panel Discussion/ Roundtables                                                                            | Bharat Mandapam / Sushma Swaraj Bhawan Ambedkar Bhawan, New Delhi |
| 20 February 2026      | GPAI Council Meeting                                                                                                | Bharat Mandapam, New Delhi                                        |
| 20 February 2026      | Keynotes/ Panel Discussion/ Roundtables                                                                             | Bharat Mandapam / Sushma Swaraj Bhawan Ambedkar Bhawan, New Delhi |
## 3.1.1.1 All farmers who have been sanctioned Seasonal Agricultural Operations

(SAO) loans from Financial Institutions (FIs) (i.e. loanee farmers) for the notified crop(s) season would be covered compulsorily. This provision shall override any decision taken by FIs including PACS exempting farmers from compulsory coverage of loanee farmers.

- 3.1.1.2 However non ­ standard KCC /crop loans as defined and as per prevailing practices of the concerned Banks/Govt. regulator shall not be covered compulsorily. However bank branches may facilitate such farmers for enrolment as non ­ loanee farmers.
- 3.1.1.3 Merely, sanctioning of crop loan against other collateral securities including fixed deposits, gold/jewel loans, mortgage loans etc. without having insurable interest of the farmer on the insurable land and notified crops shall not be covered under the Scheme.

## 3.1.2 Voluntary Component

-  The Scheme is optional for non ­ loanee farmers.
-  The insurance coverage will strictly be equivalent to sum insured /hectare, as defined in the Govt. notification or/and on National Crop Insurance Portal multiplied by sown area for notified crop.
- 3.1.3 Special efforts shall be made to ensure maximum coverage of SC/ ST/ Women farmers under the Scheme. Further Panchayat Raj Institutions (PRIs)may be involved in extension and awareness creation amongst farmers and obtaining feed ­ back of farmers about the implementation of the Scheme .
- 3.1.4 The implementing Insurance Company selected as L1 will be responsible for taking necessary measures to ensure at least 10% incremental increase in coverage of non ­ loanee farmers. However other empanelled Insurance Companies which have participated in the bidding and are keen for enrolment of non loanee farmers in the cluster may also is allowed to enrol non ­ loanee farmers at L1premium rate. The interested companies have to inform their willingness in writing within seven days of finalisation of tender/issuance of work order to L1. It will however be the responsibility of all the Insurance Companies engaged in this process


## CENTRALLY SPONSORED SCHEMES

## NATIONAL FOOD SECURITY MISSION

The scheme is being implemented from 2007 ­ 08 as a Central Sector Scheme with an objective to enhance production &amp; productivity of Rice &amp; Pulse crops in selected districts of the state in Mission Mode. Besides, this embodies Sub ­ Schemes for enhancing production &amp; productivity of Coarse Cereals under NFSM ­ Coarse Cereals, and Cotton, Jute and Sugarcane under NFSM ­ Commercial crops, NFSM ­ Nutricereals and NFSM ­ O

Same source PDF, and this time the section headings survive as real `##` Markdown headings —
Docling's layout model classifies text blocks by role (title, section-header, body text, ...)
before exporting, instead of just extracting characters and positions.


In [ ]:
docling_summit_doc = docling_converter.convert(SUMMIT_PDF).document
docling_md_summit = docling_summit_doc.export_to_markdown()

print(f"{len(docling_md_summit):,} chars")
print(f"tables detected: {len(docling_summit_doc.tables)}")


## 3. Table fidelity, side by side

`ai_summit.pdf` has exactly one real table: a five-day event schedule with **Date / Event /
Venue** columns. Same PDF, same table — here's what each tool actually produced for it.


In [ ]:
print(docling_summit_doc.tables[0].export_to_markdown(docling_summit_doc))


| Date                  | Event                                                                                                               | Venue                                                             |
|-----------------------|---------------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------|
| 16 - 20 February 2026 | AI Impact Expo                                                                                                      | Bharat Mandapam, New Delhi                                        |
| 16 February 2026      | Keynotes, Panel Discussions, Roundtables                                                                            | Bharat Mandapam/ Sushma Swaraj Bhawan/ Ambedkar Bhawan, New Delhi |
| 17 February 2026      | Release of Knowledge Compendiums on AI in Health, Energy, Education, Agriculture, Gender Empowerment, Accessibility | Bharat Mandapam, New Delhi                                        |
| 17 February 2026      | Seminar on Applied AI                                                                                               | Bharat Mandapam, New Delhi                                        |
| 17 February 2026      | AI by HER: Global Impact Challenge                                                                                  | Sushma Swaraj Bhawan, New Delhi                                   |
| 17 February 2026      | Keynotes, Panel Discussions, Roundtables                                                                            | Bharat Mandapam/ Sushma Swaraj Bhawan/ Ambedkar Bhawan, New Delhi |
| 18 February 2026      | Research Symposium                                                                                                  | Bharat Mandapam, New Delhi                                        |
| 18 February 2026      | Industry Session                                                                                                    | Bharat Mandapam, New Delhi                                        |
| 18 February 2026      | AI by HER: Global Impact Challenge                                                                                  | Sushma Swaraj Bhawan, New Delhi                                   |
| 18 February 2026      | Keynotes, Panel Discussions, Roundtables                                                                            | Bharat Mandapam/ Sushma Swaraj Bhawan/ Ambedkar Bhawan, New Delhi |
| 18 February 2026      | Summit Dinner                                                                                                       | Convention Centre, New Delhi                                      |
| 19 February 2026      | Opening Ceremony Leaders' Plenary CEO Roundtable                                                                    | Bharat Mandapam, New Delhi                                        |
| 19 February 2026      | Keynotes / Panel Discussion/ Roundtables                                                                            | Bharat Mandapam / Sushma Swaraj Bhawan Ambedkar Bhawan, New Delhi |
| 20 February 2026      | GPAI Council Meeting                                                                                                | Bharat Mandapam, New Delhi                                        |
| 20 February 2026      | Keynotes/ Panel Discussion/ Roundtables                                                                             | Bharat Mandapam / Sushma Swaraj Bhawan Ambedkar Bhawan, New Delhi |

In [ ]:
# The same table region in MarkItDown's output — find it by a string that's inside the table
idx = md_summit.rfind("Bharat Mandapam")
print(md_summit[idx - 460 : idx + 250])


Docling's dedicated table-structure model reconstructed clean rows and columns. MarkItDown's
`pdfminer`-based path **lost the table structure entirely** — the same cells come out as run-on
text with no columns, no delimiters, no way to tell "16 February 2026" apart from the event name
next to it except by the line break pdfminer happened to keep.

It gets worse in the other direction, too. Search `md_summit` for Markdown table syntax
(`` | --- `` separator rows) and there are dozens of them — but they're not real tables:


In [ ]:
print("pipe-table separator rows found in MarkItDown output:", md_summit.count("| ---"))

pipe_idx = md_summit.find("| ---")
print(md_summit[pipe_idx - 150 : pipe_idx + 300])


That's ordinary wrapped body-text paragraphs — wide word-spacing in justified text — misread as
table columns. So on this document MarkItDown fails in both directions: it loses the one table
that's actually there, and it fabricates dozens of tables where there aren't any. Docling correctly
found the one real table and rendered nothing else as a table.


## Pipeline tie-in

Either tool's Markdown output is a one-liner away from the `Document` format every other notebook
in this course expects:

```python
from langchain_core.documents import Document

doc = Document(page_content=docling_md_schemes, metadata={"source": SCHEMES_PDF, "tool": "docling"})
```

From there it's exactly the input `l22_chunking_strategies.ipynb` chunks — not re-demoed here
since chunking is already its own notebook.


## Summary

| Aspect | MarkItDown | Docling |
|---|---|---|
| Format breadth | Very wide — pptx, docx, xlsx, pdf, images, audio, and more through one API | Narrower — PDF, DOCX, PPTX, HTML, images — but each format gets real layout analysis |
| Heading structure (observed above) | Lost — flat text, no `#`/`##` markers | Preserved — real Markdown headings from a layout-role model |
| Table fidelity (observed above) | Lost the real table; fabricated ~36 fake tables from wrapped prose | Correctly extracted the one real table, cleanly, nothing else misidentified |
| LLM-enhanced mode | Yes, but pptx/image only — not PDF | No LLM step — table/layout understanding comes from a dedicated local model |
| Setup cost | Lightweight — no model download for the PDF path | Heavier — downloads/runs a local layout + table-structure model |
| Best fit | Quick, good-enough Markdown across many file types; simple prose PDFs | PDFs where headings, tables, or layout actually matter to get right |


## Takeaways

- Both tools solve "PDF to Markdown," but they solve it at different depths: MarkItDown extracts
  text with layout-preserving heuristics; Docling classifies layout with an actual model before
  exporting.
- That depth difference is invisible on simple text and glaring on structure — headings and tables
  are exactly where it showed up in this notebook's real output, not hypothetically.
- A "table-like" heuristic (column-aligned whitespace) can be worse than no table detection at
  all — MarkItDown's fabricated tables here are harder to spot as wrong than if it had just
  emitted plain paragraphs.
- Neither conversion step needed an LLM call — that only enters the pipeline later, at embedding
  and generation time, same as every other notebook in this course.
